# Quantum (QAOA) vs. classical (greedy) pruning — head-to-head

Compares the **main pipeline's** quantum result against the classical baseline:

1. `qaoa_experiment.ipynb` — the standard `qiskit_algorithms.QAOA` run (Aer-backed
   sampler) picks the pruning bitstring, saved to `qubo_outputs/qaoa_result.json`
   (includes solve timing and validation against the brute-force optimum), and its
   Step-5 real-model evaluation of exactly that bitstring, saved to
   `qubo_outputs/pruning_eval.json`.
2. `classical_pruning.ipynb` — a greedy sensitivity-ranking heuristic over
   the *same* candidate pool and *same* target compression, applied to the
   same real model and evaluated the same way, saved to
   `qubo_outputs/classical_pruning_result.json`.

> The exact-statevector mask ranking (`qaoa_ranked_masks.json` /
> `top_5_mask_recommendation.json`) is a *reference* path and is intentionally
> **not** used here — this comparison evaluates what the main QAOA run actually chose.

This notebook does not re-run any model or optimizer — it only loads those
JSON files and compares them, so it's fast and safe to re-run any time.

## Step 1 — Load results

In [1]:
import json
from pathlib import Path

OUTPUT_DIR = Path.cwd() / "qubo_outputs"

# Main-pipeline QAOA result (qaoa_experiment.ipynb, Step 4).
with (OUTPUT_DIR / "qaoa_result.json").open("r", encoding="utf-8") as f:
    qaoa = json.load(f)

# Real-model evaluation of exactly that QAOA bitstring (qaoa_experiment.ipynb, Step 5).
with (OUTPUT_DIR / "pruning_eval.json").open("r", encoding="utf-8") as f:
    quantum_eval = json.load(f)

with (OUTPUT_DIR / "classical_pruning_result.json").open("r", encoding="utf-8") as f:
    classical = json.load(f)

with (OUTPUT_DIR / "qubo_metadata.json").open("r", encoding="utf-8") as f:
    qubo_metadata = json.load(f)

classical_result = classical["result"]

# Guard: pruning_eval.json must be the evaluation OF the current qaoa_result.json
# (both are rewritten by qaoa_experiment.ipynb; a mismatch means a stale file).
assert set(quantum_eval["pruned_blocks"]) == set(qaoa["pruned_blocks"]), (
    "pruning_eval.json evaluates different blocks than qaoa_result.json selected.\n"
    f"  qaoa_result.json : {qaoa['pruned_blocks']}\n"
    f"  pruning_eval.json: {quantum_eval['pruned_blocks']}\n"
    "Re-run qaoa_experiment.ipynb end-to-end to refresh both."
)

print(f"Quantum (QAOA) : {qaoa['best_bitstring']}  -> {qaoa['pruned_blocks']}")
print(f"  validated global optimum: {qaoa['validation']['found_global_optimum']}")
print(f"Classical      : {classical_result['mask']}  -> {classical_result['pruned_blocks']}")

Quantum (QAOA) : 1100000000  -> ['stages.2.blocks.2', 'stages.3.blocks.2']
  validated global optimum: True
Classical      : 1100000000  -> stages.2.blocks.2; stages.3.blocks.2


## Step 2 — Build the side-by-side comparison table

In [2]:
quantum_selection_seconds = qaoa["qaoa_solve_seconds"]
classical_selection_seconds = classical_result["selection_time_seconds"]
# Older pruning_eval.json files did not time the evaluation; tolerate that.
quantum_eval_seconds = quantum_eval.get("eval_seconds")

rows = [
    {
        "method": "Quantum (QAOA, qiskit_algorithms + Aer sampler)",
        "mask": qaoa["best_bitstring"],
        "pruned_blocks": "; ".join(qaoa["pruned_blocks"]),
        "num_pruned_blocks": qaoa["num_pruned_blocks"],
        "predicted_compression": qaoa["total_compression"],
        "actual_parameter_reduction": quantum_eval["param_reduction_fraction"],
        "actual_accuracy": quantum_eval["pruned"]["accuracy"],
        "actual_macro_f1": quantum_eval["pruned"]["macro_f1"],
        "accuracy_drop": quantum_eval["actual"]["accuracy_drop"],
        "f1_drop": quantum_eval["actual"]["f1_drop"],
        "selection_seconds": quantum_selection_seconds,
        "eval_seconds": quantum_eval_seconds,
        "total_seconds": (quantum_selection_seconds + quantum_eval_seconds
                          if quantum_eval_seconds is not None else None),
        "n_eval_samples": quantum_eval["max_samples"],
    },
    {
        "method": "Classical (greedy sensitivity ranking)",
        "mask": classical_result["mask"],
        "pruned_blocks": classical_result["pruned_blocks"],
        "num_pruned_blocks": classical_result["num_pruned_blocks"],
        "predicted_compression": classical_result["predicted_compression"],
        "actual_parameter_reduction": classical_result["actual_parameter_reduction"],
        "actual_accuracy": classical_result["actual_accuracy"],
        "actual_macro_f1": classical_result["actual_macro_f1"],
        "accuracy_drop": classical_result["accuracy_drop"],
        "f1_drop": classical_result["f1_drop"],
        "selection_seconds": classical_selection_seconds,
        "eval_seconds": classical_result["eval_seconds"],
        "total_seconds": classical_selection_seconds + classical_result["eval_seconds"],
        "n_eval_samples": classical_result["n_samples"],
    },
]

header = (
    f"{'Method':<48} {'Blocks':>6} {'Compr.%':>8} {'ParamRed.%':>11} {'Acc':>8} "
    f"{'F1':>8} {'AccDrop':>9} {'F1Drop':>8} {'Select(s)':>11} {'Eval(s)':>9}"
)
print(header)
print("-" * len(header))
for row in rows:
    eval_str = f"{row['eval_seconds']:>9.2f}" if row["eval_seconds"] is not None else f"{'n/a':>9}"
    print(
        f"{row['method']:<48} {row['num_pruned_blocks']:>6} "
        f"{row['predicted_compression']*100:>7.2f}% {row['actual_parameter_reduction']*100:>10.2f}% "
        f"{row['actual_accuracy']*100:>7.2f}% {row['actual_macro_f1']*100:>7.2f}% "
        f"{row['accuracy_drop']*100:>8.2f}% {row['f1_drop']*100:>7.2f}% "
        f"{row['selection_seconds']:>11.6f} {eval_str}"
    )

speedup = rows[0]["selection_seconds"] / max(rows[1]["selection_seconds"], 1e-9)
print(f"\nClassical greedy selection was {speedup:,.0f}x faster than QAOA selection "
      f"({rows[1]['selection_seconds']:.6f}s vs {rows[0]['selection_seconds']:.4f}s).")

same_mask = rows[0]["mask"] == rows[1]["mask"]
print(f"Same mask chosen by both methods: {same_mask}")
print(f"QAOA bitstring verified as QUBO global optimum: "
      f"{qaoa['validation']['found_global_optimum']}")
better_f1 = "Quantum" if rows[0]["actual_macro_f1"] > rows[1]["actual_macro_f1"] else (
    "Classical" if rows[1]["actual_macro_f1"] > rows[0]["actual_macro_f1"] else "Tie")
print(f"Higher real macro-F1 after pruning: {better_f1}")

if rows[0]["n_eval_samples"] != rows[1]["n_eval_samples"]:
    print(f"\nNOTE: evaluations used different test subsets "
          f"(quantum n={rows[0]['n_eval_samples']}, classical n={rows[1]['n_eval_samples']}); "
          f"accuracy/F1 numbers are indicative, not strictly matched.")

Method                                           Blocks  Compr.%  ParamRed.%      Acc       F1   AccDrop   F1Drop   Select(s)   Eval(s)
---------------------------------------------------------------------------------------------------------------------------------------
Quantum (QAOA, qiskit_algorithms + Aer sampler)       2   21.43%      21.43%   96.00%   95.74%     3.17%    3.37%    3.642723     81.93
Classical (greedy sensitivity ranking)                2   21.43%      21.43%   96.00%   95.74%     3.17%    3.37%    0.000568     79.42

Classical greedy selection was 6,410x faster than QAOA selection (0.000568s vs 3.6427s).
Same mask chosen by both methods: True
QAOA bitstring verified as QUBO global optimum: True
Higher real macro-F1 after pruning: Tie


## Step 3 — Does quantum show a benefit here, and does that generalize "at scale"?

**Quality of the decision.** Compare `actual_macro_f1` / `accuracy_drop` above.
The QAOA bitstring is validated against the classical brute force
(`qaoa_result.json → validation.found_global_optimum`), so when QAOA outperforms
the greedy heuristic here, it's really the *QUBO objective itself* (loss/compression
trade-off with topology + budget-guard terms) winning, not "quantum" in the
computational sense — a classical solver minimizing the identical QUBO would
land on the identical mask and get the identical real-world result.

**Speed of the decision.** This is the part that actually says something about
*quantum vs. classical as computational methods*, and here the numbers above
should be read carefully:

- The classical greedy heuristic is a single sort plus a running sum over
  `n` candidates — microseconds, and it stays microseconds as `n` grows into
  the thousands.
- The "QAOA" timing measured above is **not a real quantum computer** — it's
  `qiskit-aer` simulating the QAOA gate circuit on a classical CPU (hundreds of
  COBYLA iterations × shots per iteration). Simulating an `n`-qubit circuit
  still costs exponentially more as `n` grows *no matter how good the QAOA
  angles are*: at `n=10` a state is ~1024 amplitudes (cheap); at `n=30` it's
  ~10^9 (already impractical on a laptop); at `n=50`+ it is not simulable at
  all on classical hardware.

So this benchmark cannot honestly answer "does quantum have an edge at scale"
— it can only show that, **on a classical computer simulating a small quantum
circuit, the classical heuristic is currently far faster**, which is expected
and not a meaningful verdict on quantum hardware. The only way to actually
test the scaling question is to run the QAOA circuit on **real quantum
hardware** (larger qubit counts, more candidates) and compare *that* wall
clock against classical solvers of the same QUBO at the same size — the
Aer-simulated numbers above are a correctness/quality check on the QUBO
formulation, not a hardware performance benchmark.

## Step 4 — Save the comparison table

In [3]:
import csv

fieldnames = list(rows[0].keys())
comparison_csv = OUTPUT_DIR / "quantum_vs_classical_comparison.csv"

with comparison_csv.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for row in rows:
        writer.writerow(row)

comparison_json = OUTPUT_DIR / "quantum_vs_classical_comparison.json"
summary = {
    "quantum_source": "qaoa_result.json + pruning_eval.json (main pipeline, qaoa_experiment.ipynb)",
    "classical_source": "classical_pruning_result.json (classical_pruning.ipynb)",
    "rows": rows,
    "same_mask_chosen": same_mask,
    "qaoa_found_global_optimum": qaoa["validation"]["found_global_optimum"],
    "quantum_selection_seconds": quantum_selection_seconds,
    "classical_selection_seconds": classical_selection_seconds,
    "classical_speedup_factor_over_qaoa_selection": speedup,
    "caveat": (
        "QAOA timing is from a classical qiskit-aer gate-circuit simulation "
        "(exponential cost in qubit count), not real quantum hardware. This comparison "
        "shows decision quality parity/difference and simulated-QAOA overhead, not real "
        "hardware scaling behavior. Quantum and classical evaluations may also use "
        "different test-subset sizes (see n_eval_samples per row)."
    ),
}

with comparison_json.open("w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Saved:", comparison_csv)
print("Saved:", comparison_json)

Saved: C:\_projects\quantum_pruning\qubo_outputs\quantum_vs_classical_comparison.csv
Saved: C:\_projects\quantum_pruning\qubo_outputs\quantum_vs_classical_comparison.json
